In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [21]:
# Reproducibility
seed = 42

np.random.seed(seed)
tf.keras.utils.set_random_seed(seed)

# CREATE THE DATASET

In [22]:
# We generate 2000 examples for a binary classification task

In [23]:
X, y = make_moons(n_samples=2000, noise=0.25, random_state=seed)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

In [25]:
# Scale the features using training data only

scaler = StandardScaler()

In [26]:
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

# Define Initialization methods

In [27]:
initializers = {
    "Xavier Normal": tf.keras.initializers.GlorotNormal,
    "Xavier Uniform":tf.keras.initializers.GlorotUniform ,
    "Glorot Normal": tf.keras.initializers.HeNormal,
    "Glorot Uniform": tf.keras.initializers.HeUniform
}

# Function to build a model

In [28]:
def build_model(initializer_class):

    initializer = initializer_class(seed=seed)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),

        tf.keras.layers.Dense(
            64,
            activation="relu",
            kernel_initializer=initializer,
            bias_initializer="zero"
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu",
            kernel_initializer=initializer_class(seed=seed + 1),
            bias_initializer="zero"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            kernel_initializer=initializer_class(seed=seed + 2),
            bias_initializer="zero"
        )
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])

    return model

# Experiment with each initializer

In [29]:
results = []

for name, initializer_class in initializers.items():

    print(f"\nTraining with {name}...")

    tf.keras.backend.clear_session()

    # Build a fresh model for every experiment
    model = build_model(initializer_class)

    # Measure initial predictions and activations
    first_layer = tf.keras.Model(
        inputs=model.inputs,
        outputs=model.layers[0].output
    )

    second_layer = tf.keras.Model(
        inputs=model.inputs,
        outputs=model.layers[1].output
    )

    activation_1 = first_layer.predict(
        X_train, verbose=0
    )

    activation_2 = second_layer.predict(
        X_train, verbose=0
    )

    initial_loss = model.evaluate(
        X_train, y_train, verbose=0
    )[0]

    # Train the network
    history = model.fit(
        X_train,
        y_train,
        epochs=60,
        batch_size=32,
        verbose=0
    )

    # Evaluate on unseen test data
    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    results.append({
        "Initializer": name,
        "Layer 1 Activation Std": np.std(activation_1),
        "Layer 2 Activation Std": np.std(activation_2),
        "Initial Loss": initial_loss,
        "Final Training Loss": history.history["loss"][-1],
        "Test Loss": test_loss,
        "Test Accuracy (%)": test_accuracy * 100
    })


Training with Xavier Normal...

Training with Xavier Uniform...

Training with Glorot Normal...

Training with Glorot Uniform...


# Display the results

In [30]:
results_df = pd.DataFrame(results)

print("\nWEIGHT INITIALIZATION COMPARISON")
print(results_df.round(4).to_string(index=False))



WEIGHT INITIALIZATION COMPARISON
   Initializer  Layer 1 Activation Std  Layer 2 Activation Std  Initial Loss  Final Training Loss  Test Loss  Test Accuracy (%)
 Xavier Normal                  0.1370                  0.0875        0.7016               0.1385     0.1831              92.50
Xavier Uniform                  0.1427                  0.0952        0.6863               0.1380     0.1884              91.75
 Glorot Normal                  0.7873                  0.7112        0.7851               0.1331     0.1903              92.75
Glorot Uniform                  0.8195                  0.7731        0.8424               0.1348     0.1847              92.50
